In [0]:
BOOTSTRAP_SERVERS='pkc-56d1g.eastus.azure.confluent.cloud:9092'
TOPIC_NAME='credit-card-transaction'
API_KEY='EGUCJGGISVW6FQXZ'
API_SECRET='cfltZ5mqVTG9GjzSGLaKaKNFAe5/lQaT5xbEazK+ZzDV/MYbvaMm9Ntij3cy0+cg'

In [0]:
df = spark.readStream.format('kafka')\
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)\
    .option("subscribe", TOPIC_NAME)\
    .option("kafka.security.protocol", "SASL_SSL")\
    .option("kafka.sasl.mechanism", "PLAIN")\
    .option("kafka.sasl.jaas.config", f"""kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";""")\
    .option("startingOffsets", "earliest")\
    .load()

In [0]:
from pyspark.sql.functions import col
df = df.select(col('key').cast('string').alias('key')\
              , col('value').cast('string').alias('value')
              , col('topic'), col('partition'), col('offset')
              , col('timestamp'))

In [0]:
sQuery =df.writeStream\
            .outputMode('append')\
            .option('checkpointLocation', '/Volumes/fraud_detection/source/checkpointlocation/transaction/')\
            .trigger(availableNow=True)\
            .toTable('fraud_detection.bronze.transactions')

In [0]:
%sql
select distinct ingestion_time from fraud_detection.bronze.transactions_dp

In [0]:
%sql
select count(*) from fraud_detection.silver.customers

In [0]:
%sql
select * from fraud_detection.gold.high_value_transaction_alert

In [0]:
%sql
select * from fraud_detection.gold.fraud_card_alert

In [0]:
%sql
SELECT * FROM fraud_detection.gold.fraud_card_alert

In [0]:
%sql
SELECT * FROM fraud_detection.bronze.fraud_watchlist_dp

In [0]:
%sql
SELECT * FROM fraud_detection.silver.fraud_watchlist_dp

In [0]:
%sql
SELECT * FROM fraud_detection.gold.transactions_per_minute